# Notebook: case_2_analysis
# Purpose: Analyze X (Twitter) tweets about MBG and prepare slide-ready outputs

# Sections:
# 1. Imports & setup
# 2. Load dataset
# 3. Data validation & cleaning
# 4. Text preprocessing
# 5. EDA
# 6. Topic modeling (LDA + BERTopic fallback)
# 7. Sentiment & polarization
# 8. Network analysis
# 9. Temporal analysis
# 10. Export artifacts & slide content

print('Notebook scaffold created. Follow cells sequentially.')

In [1]:
# %%
# Section 1: Install and import required packages (run once)
# If running in an environment where packages are missing, uncomment the pip installs.

%pip install pandas numpy openpyxl matplotlib seaborn plotly scikit-learn gensim sentence-transformers bertopic networkx python-louvain python-pptx nltk wordcloud spacy Sastrawi

import os
import random
from pathlib import Path

# Paths
# Use Kaggle working directory when available; otherwise fall back to the current working folder.
KAGGLE_WORKING_DIR = Path('/kaggle/working')
BASE_DIR = KAGGLE_WORKING_DIR if KAGGLE_WORKING_DIR.exists() else Path.cwd()
# Dataset directory (Kaggle input)
DATASET_DIR = Path('/kaggle/input/datasets/zkyfauzi/case2bdc') if Path('/kaggle/input/datasets/zkyfauzi/case2bdc').exists() else BASE_DIR
# File paths
DATA_PATH = DATASET_DIR / 'case_2_dataset.xlsx'
TEMPLATE_PATH = DATASET_DIR / 'case_2_template.pptx'
# Output directory
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

print('DATA_PATH =', DATA_PATH)
print('TEMPLATE_PATH =', TEMPLATE_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 12.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
DATA_PATH = /kaggle/input/datasets/zkyfauzi/case2bdc/case_2_dataset.xlsx
TEMPLATE_PATH = /kaggle/input/datasets/zkyfauzi/case2bdc/case_2_template.pptx
OUTPUT_DIR = /kaggle/working/outputs


In [2]:
# %%
# Section 1b: Imports and environment checks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import json

print('pandas', pd.__version__)
print('numpy', np.__version__)
print('matplotlib', plt.matplotlib.__version__)
print('seaborn', sns.__version__)
print('networkx', nx.__version__)

pandas 2.3.3
numpy 2.0.2
matplotlib 3.10.0
seaborn 0.13.2
networkx 3.6.1


In [3]:
# %%
# Section 2: Load dataset (.xlsx) and quick inspection
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

# Attempt to read all sheets and guess which sheet contains tweets
xls = pd.ExcelFile(DATA_PATH)
print('Sheets:', xls.sheet_names)

# Heuristic: choose first sheet
df = pd.read_excel(xls, sheet_name=0, engine='openpyxl')

print('Rows, cols:', df.shape)
print('Columns:', df.columns.tolist())

df.head().T


Sheets: ['Sheet1']
Rows, cols: (15000, 32)
Columns: ['date', 'lang', 'source', 'user_id', 'hashtags', 'is_quote', 'mentions', 'tweet_id', 'username', 'verified', 'full_text', 'created_at', 'quoted_url', 'view_count', 'quote_count', 'quoted_text', 'reply_count', 'display_name', 'retweet_count', 'user_location', 'favorite_count', 'in_reply_to_url', 'quoted_tweet_id', 'quoted_username', 'user_created_at', 'user_description', 'in_reply_to_user_id', 'user_statuses_count', 'user_followers_count', 'user_following_count', 'in_reply_to_status_id', 'in_reply_to_screen_name']


,0,1,2,3,4
date,45662.580625,45752.56706,45671.931933,45715.643438,45946.966262
lang,in,in,in,in,in
source,dlvr.it,Twitter Web App,Twitter for Android,Twitter for Android,Twitter for Android
user_id,1666303982007111680,1356909896738820096,553451866,1300748401890320384,1910040869513879552
hashtags,[],"[""PenuhiGiziIndonesia"",""Indonesiaemas"",""AstaCi...",[],[],[]
is_quote,False,False,False,False,False
mentions,[],[],"[{""name"":""Lambe Waras"",""user_id"":""2905814730"",...","[{""name"":""Tanyarl 💚"",""user_id"":""13316505595189...",[]
tweet_id,1875904106541559808,1908514097764897024,1879292908261794304,1895133428124918272,1978962010009223168
username,arinadotid,ArifaNidya,ZainAris,hereseans,revirevii1
verified,False,False,False,False,False


In [4]:
# %%
# Section 3: Data schema validation, missing values & filtering
required_cols = ['tweet_id', 'text', 'user_id', 'created_at']
# Attempt to map likely columns if names differ
cols_lower = [c.lower() for c in df.columns]
print('lowercase cols sample:', cols_lower[:20])

# Find matches for required cols
col_map = {}
for rc in required_cols:
    for c in df.columns:
        if rc in c.lower():
            col_map[rc] = c
            break

print('Column mapping (detected):', col_map)

# If exact names present, use them; otherwise, try to proceed with heuristics
if 'tweet_id' not in col_map:
    # try id-like
    for c in df.columns:
        if 'id' in c.lower():
            col_map.setdefault('tweet_id', c)
            break

if 'text' not in col_map:
    for c in df.columns:
        if 'text' in c.lower() or 'tweet' in c.lower() or 'body' in c.lower():
            col_map.setdefault('text', c)
            break

if 'created_at' not in col_map:
    for c in df.columns:
        if 'time' in c.lower() or 'date' in c.lower():
            col_map.setdefault('created_at', c)
            break

print('Final column map:', col_map)

# Create standard columns
df = df.rename(columns={v:k for k,v in col_map.items()})

# Keep only relevant columns if available
keep_cols = [c for c in ['tweet_id','text','user_id','created_at','in_reply_to','retweet_of','quote_of','language'] if c in df.columns]
print('Keeping columns:', keep_cols)

df = df[keep_cols].copy()

# Convert created_at to datetime where possible
if 'created_at' in df.columns:
    df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
    print('created_at nulls:', df['created_at'].isna().sum())

# Drop rows without text
before = len(df)
df = df[df['text'].notna()]
after = len(df)
print('Dropped empty-text rows:', before-after)

# Remove exact duplicates based on text + user + created_at
df = df.drop_duplicates(subset=['text','user_id','created_at'])
print('After dedup shape:', df.shape)

# Save cleaned CSV for traceability
clean_path = OUTPUT_DIR / 'cleaned_dataset.csv'
df.to_csv(clean_path, index=False)
print('Cleaned dataset saved to', clean_path)


lowercase cols sample: ['date', 'lang', 'source', 'user_id', 'hashtags', 'is_quote', 'mentions', 'tweet_id', 'username', 'verified', 'full_text', 'created_at', 'quoted_url', 'view_count', 'quote_count', 'quoted_text', 'reply_count', 'display_name', 'retweet_count', 'user_location']
Column mapping (detected): {'tweet_id': 'tweet_id', 'text': 'full_text', 'user_id': 'user_id', 'created_at': 'created_at'}
Final column map: {'tweet_id': 'tweet_id', 'text': 'full_text', 'user_id': 'user_id', 'created_at': 'created_at'}
Keeping columns: ['tweet_id', 'text', 'user_id', 'created_at']


/tmp/ipykernel_23/3700613282.py:51: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')


created_at nulls: 0
Dropped empty-text rows: 0
After dedup shape: (14999, 4)
Cleaned dataset saved to /kaggle/working/outputs/cleaned_dataset.csv


In [5]:
# %%
# Section 4: Text preprocessing: cleaning & normalization
import re
import html
from nltk.corpus import stopwords

# Download NLTK data if needed
import nltk
nltk.download('stopwords')

# Basic cleaning functions
URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#\w+')
RT_RE = re.compile(r'\bRT\b')
EMOJI_RE = re.compile("[\U00010000-\U0010ffff]", flags=re.UNICODE)

ind_stopwords = set()
try:
    ind_stopwords = set(stopwords.words('indonesian'))
except Exception:
    # fallback: use english set and small custom list
    ind_stopwords = set(stopwords.words('english'))
    ind_stopwords.update({'yg','dari','dan','ke','yg','ini','itu'})


def clean_text(s, keep_hashtags=False):
    if not isinstance(s, str):
        return ''
    s = html.unescape(s)
    s = URL_RE.sub('', s)
    s = MENTION_RE.sub('', s)
    if not keep_hashtags:
        s = HASHTAG_RE.sub('', s)
    s = RT_RE.sub('', s)
    s = EMOJI_RE.sub('', s)
    s = re.sub(r'[^\w\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.lower()
    return s

# Apply cleaning
df['clean_text'] = df['text'].astype(str).apply(lambda x: clean_text(x, keep_hashtags=True))

# Tokenization simple split
df['tokens'] = df['clean_text'].str.split()

# Tokenized minus stopwords
df['tokens_nostop'] = df['tokens'].apply(lambda toks: [t for t in toks if t not in ind_stopwords and len(t)>1])

# Save tokenized sample
df[['tweet_id','clean_text']].head(10).to_csv(OUTPUT_DIR / 'sample_clean_text.csv', index=False)
print('Preprocessing complete. Sample saved.')


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Preprocessing complete. Sample saved.


In [6]:
# %%
# Section 7: Exploratory Data Analysis
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

# Time series volume
if 'created_at' in df.columns:
    ts = df.set_index('created_at').resample('D').size()
    plt.figure(figsize=(10,4))
    sns.lineplot(x=ts.index, y=ts.values)
    plt.title('Daily tweet volume')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'daily_volume.png', dpi=200)
    plt.close()

# Top users by count
if 'user_id' in df.columns:
    top_users = df['user_id'].value_counts().head(20)
    top_users.to_csv(OUTPUT_DIR / 'top_users.csv')

# Top hashtags
all_hashtags = Counter()
for text in df['text'].astype(str):
    tags = HASHTAG_RE.findall(text)
    all_hashtags.update([t.lower() for t in tags])

top_hashtags = all_hashtags.most_common(30)
pd.DataFrame(top_hashtags, columns=['hashtag','count']).to_csv(OUTPUT_DIR / 'top_hashtags.csv', index=False)

# Top n-grams
cv = CountVectorizer(ngram_range=(1,2), max_features=2000)
X = cv.fit_transform(df['clean_text'].astype(str))
sum_words = X.sum(axis=0)
words_freq = [(word, sum_words[0, idx]) for word, idx in cv.vocabulary_.items()]
words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
pd.DataFrame(words_freq[:50], columns=['ngram','count']).to_csv(OUTPUT_DIR / 'top_ngrams.csv', index=False)

print('EDA artifacts saved to outputs/')

EDA artifacts saved to outputs/


In [7]:
# %%
# Section 10: Topic modeling - LDA pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Prepare CountVectorizer
n_topics = 10
vectorizer = CountVectorizer(max_df=0.95, min_df=5, max_features=20000, ngram_range=(1,2))
X_counts = vectorizer.fit_transform(df['clean_text'].astype(str))

lda = LatentDirichletAllocation(n_components=n_topics, random_state=RANDOM_SEED, n_jobs=-1)
lda.fit(X_counts)

# Extract topics
def lda_top_terms(model, feature_names, n_top=12):
    topics = {}
    for i, topic in enumerate(model.components_):
        top_idx = topic.argsort()[:-n_top - 1:-1]
        topics[i] = [feature_names[j] for j in top_idx]
    return topics

feature_names = vectorizer.get_feature_names_out()
topics = lda_top_terms(lda, feature_names, n_top=12)

with open(OUTPUT_DIR / 'lda_topics.json', 'w', encoding='utf-8') as f:
    json.dump(topics, f, ensure_ascii=False, indent=2)

print('LDA topics saved. Example topic 0:', topics.get(0)[:10])

# BERTopic fallback (pseudocode): if BERTopic installed, run it on sentence-transformers embeddings
try:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    emb_model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = emb_model.encode(df['clean_text'].astype(str).tolist(), show_progress_bar=True)
    topic_model = BERTopic(min_topic_size=50, verbose=True)
    topics_bt, probs = topic_model.fit_transform(df['clean_text'].astype(str).tolist(), embeddings)
    topic_model.save(OUTPUT_DIR / 'bertopic_model')
    print('BERTopic finished with', len(set(topics_bt)), 'topics')
except Exception as e:
    print('BERTopic not run:', e)

# Assign dominant LDA topic to each doc
lda_topics_per_doc = lda.transform(X_counts).argmax(axis=1)
df['lda_topic'] = lda_topics_per_doc

df[['tweet_id','lda_topic']].to_csv(OUTPUT_DIR / 'tweet_topics.csv', index=False)

LDA topics saved. Example topic 0: ['mbg', 'dan', 'di', 'yg', 'negara', 'untuk', 'orang', 'ini', 'siswa', 'dari']


2026-05-16 15:59:06.555433: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778947146.763554      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778947146.829017      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778947147.340015      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778947147.340058      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778947147.340061      23 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

2026-05-16 16:00:13,538 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-16 16:00:40,815 - BERTopic - Dimensionality - Completed ✓
2026-05-16 16:00:40,819 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-16 16:00:41,909 - BERTopic - Cluster - Completed ✓
2026-05-16 16:00:41,919 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-16 16:00:42,205 - BERTopic - Representation - Completed ✓
2026-05-16 16:00:42,357 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


BERTopic finished with 29 topics


In [8]:
# %%
# Section 13-16: Sentiment & Network analysis (basic)
# Sentiment - simple multilingual fallback using TextBlob (or VADER for English)
try:
    from textblob import TextBlob
    def polarity_score(text):
        return TextBlob(text).sentiment.polarity
    df['polarity'] = df['clean_text'].astype(str).apply(polarity_score)
    df['sentiment_label'] = df['polarity'].apply(lambda x: 'pos' if x>0.05 else ('neg' if x<-0.05 else 'neu'))
    df[['tweet_id','polarity','sentiment_label']].to_csv(OUTPUT_DIR / 'sentiment_basic.csv', index=False)
    print('Basic sentiment computed (TextBlob)')
except Exception as e:
    print('TextBlob not available fallback:', e)

# Network construction: mentions and retweets
G = nx.DiGraph()

# Mentions
for _, row in df.iterrows():
    sender = str(row.get('user_id', 'unknown'))
    text = str(row.get('text',''))
    mentions = re.findall(r'@([A-Za-z0-9_]+)', text)
    for m in mentions:
        target = m
        G.add_edge(sender, target, interaction='mention')

# If retweet info available
if 'retweet_of' in df.columns:
    for _, row in df[df['retweet_of'].notna()].iterrows():
        sender = str(row.get('user_id','unknown'))
        target = str(row.get('retweet_of'))
        if not G.has_edge(sender,target):
            G.add_edge(sender,target, interaction='retweet', weight=1)
        else:
            G[sender][target]['weight'] = G[sender][target].get('weight',1)+1

# Compute centralities for nodes
deg = dict(G.degree())
bet = nx.betweenness_centrality(G)

central_df = pd.DataFrame([{'user':u,'degree':deg.get(u,0),'betweenness':bet.get(u,0.0)} for u in G.nodes()])
central_df.sort_values('degree', ascending=False).head(20).to_csv(OUTPUT_DIR / 'top_influencers.csv', index=False)

# Save network graph (GEXF) for interactive visualization
nx.write_gexf(G, OUTPUT_DIR / 'interaction_graph.gexf')
print('Network artifacts saved.')


Basic sentiment computed (TextBlob)
Network artifacts saved.


In [9]:
# %%
# Section 21: Generate slide content snippets mapped to template
slides = {
    'background': {
        'title': 'Background',
        'bullets': [
            'Digital discussion around MBG on platform X reflects public sentiment and debate.',
            'Dataset: tweets collected during program implementation period (provided by panitia).',
            'Objective: analyze topics, actors, polarization, and actionable recommendations.'
        ],
        'figures': [str(OUTPUT_DIR / 'daily_volume.png')]
    },
    'research_question': {
        'title': 'Research Questions',
        'bullets': [
            'What are the main themes in public discussion about MBG?',
            'Which actors shape the conversation and how information flows?',
            'How polarized is the public discourse and which topics drive polarization?'
        ],
        'figures': []
    },
    'methodology': {
        'title': 'Methodology (1/2)',
        'bullets': [
            'Data cleaning and preprocessing of provided tweets.',
            'Topic modeling (LDA, BERTopic fallback) to identify themes.'
        ],
        'figures': []
    },
    'methodology_2': {
        'title': 'Methodology (2/2)',
        'bullets': [
            'Network analysis to detect influencers and communities.',
            'Sentiment & polarization analysis aggregated by topic and time.'
        ],
        'figures': []
    },
    'results': {
        'title': 'Results & Discussion',
        'bullets': [],
        'figures': [str(OUTPUT_DIR / 'top_hashtags.csv'), str(OUTPUT_DIR / 'top_influencers.csv')]
    },
    'important_findings': {
        'title': 'Important Findings',
        'bullets': [
            'Top themes identified with prevalence percentages.',
            'Key influencer accounts and their roles in information spread.',
            'Measured level of polarity and topics with highest divergence.'
        ],
        'figures': []
    },
    'suggestions': {
        'title': 'Suggestions',
        'bullets': [
            'Design communication addressing top concerns and misinformation channels.',
            'Engage influencers from supportive and neutral communities for balanced outreach.',
            'Monitor topic evolution and rapid response for spikes in negative sentiment.'
        ],
        'figures': []
    },
    'ai_declaration': {
        'title': 'AI Usage Declaration',
        'bullets': [
            'AI tools used for code scaffolding and literature guidance only (to be declared).',
            'All substantive analysis, modeling decisions, and final interpretations performed by the team.'
        ],
        'figures': []
    }
}

with open(OUTPUT_DIR / 'slide_contents.json', 'w', encoding='utf-8') as f:
    json.dump(slides, f, ensure_ascii=False, indent=2)

print('Slide contents JSON written to outputs/slide_contents.json')


Slide contents JSON written to outputs/slide_contents.json


In [10]:
# %%
# Section 5b: Domain-specific feature engineering and stronger stance/sentiment proxy
DOMAIN_STOPWORDS = {
    'mbg', 'makan', 'bergizi', 'gratis', 'program', 'makanbergizigratis', 'programmbg',
    'makanbergizi', 'bergizigratis', 'programmakanbergizi', 'gratismbg', 'yg', 'yang', 'dan',
    'di', 'ke', 'dari', 'untuk', 'itu', 'ini', 'aja', 'gak', 'ga', 'ya', 'nya', 'lah', 'pun',
    'rt', 'amp', 'https', 'co'
}


def detect_stance(text):
    t = str(text).lower()
    support_markers = ['dukung', 'lanjutkan', 'manfaat', 'hak', 'baik', 'sehat', 'positif', 'sukses', 'berdampak']
    oppose_markers = ['stop', 'keracunan', 'racun', 'boros', 'anggaran', 'kritik', 'gagal', 'batal', 'tolak', 'bancakan']
    support = sum(marker in t for marker in support_markers)
    oppose = sum(marker in t for marker in oppose_markers)
    if support > oppose:
        return 'support'
    if oppose > support:
        return 'oppose'
    return 'neutral'


df['tokens_domain'] = df['tokens_nostop'].apply(lambda toks: [t for t in toks if t not in DOMAIN_STOPWORDS])
df['token_count'] = df['tokens_domain'].apply(len)
df['char_count'] = df['clean_text'].str.len()
df['hashtag_count'] = df['text'].astype(str).apply(lambda s: len(HASHTAG_RE.findall(s)))
df['mention_count'] = df['text'].astype(str).apply(lambda s: len(re.findall(r'@\w+', s)))
df['url_count'] = df['text'].astype(str).apply(lambda s: len(URL_RE.findall(s)))
df['stance_proxy'] = df['clean_text'].apply(detect_stance)

try:
    from transformers import pipeline
    sentiment_pipe = pipeline('sentiment-analysis', model='cardiffnlp/twitter-xlm-roberta-base-sentiment', truncation=True)
    sample_size = min(len(df), 2000)
    sentiment_results = sentiment_pipe(df['clean_text'].astype(str).tolist()[:sample_size])
    label_map = {'positive': 1.0, 'neutral': 0.0, 'negative': -1.0}
    scores = [label_map.get(r['label'].lower(), 0.0) * float(r['score']) for r in sentiment_results]
    df.loc[df.index[:sample_size], 'sentiment_score'] = scores
    if 'sentiment_score' not in df.columns:
        df['sentiment_score'] = 0.0
    df['sentiment_score'] = df['sentiment_score'].fillna(0.0)
    df['sentiment_class'] = pd.cut(df['sentiment_score'], bins=[-np.inf, -0.15, 0.15, np.inf], labels=['negative', 'neutral', 'positive'])
except Exception as e:
    df['sentiment_score'] = df['polarity'] if 'polarity' in df.columns else 0.0
    df['sentiment_class'] = pd.cut(df['sentiment_score'], bins=[-np.inf, -0.05, 0.05, np.inf], labels=['negative', 'neutral', 'positive'])
    print('Transformer sentiment unavailable, fallback used:', e)

feature_preview = df[['tweet_id', 'token_count', 'char_count', 'hashtag_count', 'mention_count', 'url_count', 'stance_proxy', 'sentiment_score', 'sentiment_class']].head()
feature_preview

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

,tweet_id,token_count,char_count,hashtag_count,mention_count,url_count,stance_proxy,sentiment_score,sentiment_class
0,1875904106541559808,12,123,0,0,1,neutral,0.000000,neutral
1,1908514097764897024,8,126,6,0,1,support,0.605242,positive
2,1879292908261794304,15,207,0,2,0,neutral,0.000000,neutral
3,1895133428124918272,1,8,0,2,0,neutral,0.000000,neutral
4,1978962010009223168,14,159,0,0,1,support,0.000000,neutral


In [11]:
# %%
# Section 11b: Semantic topic modeling with embeddings (BERTopic or fallback clustering)
topic_texts = df['clean_text'].astype(str).tolist()
semantic_topics = None
semantic_model_path = OUTPUT_DIR / 'bertopic_model'

try:
    from sentence_transformers import SentenceTransformer
    from bertopic import BERTopic

    embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    embeddings = embedder.encode(topic_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    semantic_model = BERTopic(min_topic_size=25, n_gram_range=(1, 2), calculate_probabilities=True, verbose=True)
    semantic_topics, semantic_probs = semantic_model.fit_transform(topic_texts, embeddings)
    semantic_model.save(str(semantic_model_path))
    semantic_info = semantic_model.get_topic_info()
    semantic_info.to_csv(OUTPUT_DIR / 'bertopic_topic_info.csv', index=False)
    print(semantic_info.head())
except Exception as e:
    print('BERTopic unavailable, fallback to TF-IDF clustering:', e)
    from sklearn.cluster import MiniBatchKMeans

    tfidf_sem = TfidfVectorizer(max_df=0.9, min_df=3, ngram_range=(1, 2), max_features=30000)
    X_sem = tfidf_sem.fit_transform(topic_texts)
    kmeans = MiniBatchKMeans(n_clusters=10, random_state=RANDOM_SEED, batch_size=256)
    semantic_topics = kmeans.fit_predict(X_sem)
    terms = np.array(tfidf_sem.get_feature_names_out())
    cluster_terms = {}
    for cluster_id in sorted(set(semantic_topics)):
        center = kmeans.cluster_centers_[cluster_id]
        cluster_terms[int(cluster_id)] = terms[np.argsort(center)[::-1][:12]].tolist()
    with open(OUTPUT_DIR / 'semantic_topic_terms.json', 'w', encoding='utf-8') as f:
        json.dump(cluster_terms, f, ensure_ascii=False, indent=2)

if semantic_topics is not None:
    df['semantic_topic'] = semantic_topics
    df[['tweet_id', 'semantic_topic']].to_csv(OUTPUT_DIR / 'semantic_topics.csv', index=False)

print('Semantic topic artifacts saved.')


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/235 [00:00<?, ?it/s]

2026-05-16 16:06:03,171 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-16 16:06:09,210 - BERTopic - Dimensionality - Completed ✓
2026-05-16 16:06:09,212 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-16 16:06:12,201 - BERTopic - Cluster - Completed ✓
2026-05-16 16:06:12,207 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-16 16:06:13,411 - BERTopic - Representation - Completed ✓
2026-05-16 16:06:13,800 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


   Topic  Count                                     Name  \
0     -1   5444                 -1_mbg_program_makan_dan   
1      0   1828                     0_mbg_mbg mbg_aja_ga   
2      1    860            1_sekolah_pendidikan_mbg_guru   
3      2    720  2_indonesia_makanbergizigratis_mbg_anak   
4      3    704                 3_anggaran_mbg_dana_buat   

                                      Representation  \
0  [mbg, program, makan, dan, yg, gratis, di, ber...   
1  [mbg, mbg mbg, aja, ga, ya, yg, ini, gw, nya, di]   
2  [sekolah, pendidikan, mbg, guru, yg, di, ada, ...   
3  [indonesia, makanbergizigratis, mbg, anak, pro...   
4  [anggaran, mbg, dana, buat, yg, duit, pajak, i...   

                                 Representative_Docs  
0  [program makan bergizi gratis demi gizi anak a...  
1      [mana mbg nya, gak dapet mbg ya pak, mbg nya]  
2  [smoga ada nanti pendidikan gratis karena prog...  
3  [mendukung program mbg makanbergizigratis manf...  
4  [tapi kalo anies naik ga

In [12]:
# %%
# Section 14b-15b: Community detection, polarization, and anomaly detection
import warnings

topic_col = 'semantic_topic' if 'semantic_topic' in df.columns else 'lda_topic'

user_graph = nx.Graph()

for _, row in df.iterrows():
    sender = str(row.get('user_id', 'unknown'))
    text = str(row.get('text', ''))
    mentions = re.findall(r'@([A-Za-z0-9_]+)', text)
    for target in mentions:
        target = target.lower()
        if sender == target:
            continue
        if user_graph.has_edge(sender, target):
            user_graph[sender][target]['weight'] += 1
        else:
            user_graph.add_edge(sender, target, weight=1)

try:
    import importlib
    community_louvain = importlib.import_module('community')
    partition = community_louvain.best_partition(user_graph, weight='weight', random_state=RANDOM_SEED)
except Exception:
    partition = {n: i for i, n in enumerate(user_graph.nodes())}

if partition:
    community_df = pd.DataFrame({'user': list(partition.keys()), 'community': list(partition.values())})
    community_df.to_csv(OUTPUT_DIR / 'communities.csv', index=False)
    df['user_community'] = df['user_id'].astype(str).map(partition)

community_sentiment = pd.DataFrame()
if 'user_community' in df.columns:
    community_sentiment = df.groupby('user_community')['sentiment_score'].agg(['mean', 'median', 'count']).reset_index()
    community_sentiment.to_csv(OUTPUT_DIR / 'community_sentiment.csv', index=False)

if user_graph.number_of_nodes() > 1 and 'user_community' in df.columns and df['user_community'].nunique() > 1:
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            assortativity = nx.attribute_assortativity_coefficient(user_graph, 'community')
        if np.isnan(assortativity):
            assortativity = None
    except Exception:
        assortativity = None
else:
    assortativity = None

polarization_report = {
    'community_assortativity': None if assortativity is None else float(assortativity),
    'community_count': int(df['user_community'].nunique()) if 'user_community' in df.columns else 0,
    'support_vs_oppose_sentiment_gap': float(df.loc[df['stance_proxy'] == 'support', 'sentiment_score'].mean() - df.loc[df['stance_proxy'] == 'oppose', 'sentiment_score'].mean()) if {'support', 'oppose'}.issubset(set(df['stance_proxy'].unique())) else None,
}
with open(OUTPUT_DIR / 'polarization_report.json', 'w', encoding='utf-8') as f:
    json.dump(polarization_report, f, ensure_ascii=False, indent=2)

# User-level anomaly detection as proxy for coordinated amplification / buzzer-like activity
user_features = df.groupby('user_id').agg({
    'tweet_id': 'count',
    'token_count': 'mean',
    'hashtag_count': 'mean',
    'mention_count': 'mean',
    'url_count': 'mean',
    'sentiment_score': 'mean',
}).rename(columns={'tweet_id': 'tweet_volume'})
user_features['unique_topic_count'] = df.groupby('user_id')[topic_col].nunique()
user_features['stance_entropy'] = df.groupby('user_id')['stance_proxy'].apply(
    lambda s: float(-(p := s.value_counts(normalize=True)).mul(np.log2(p + 1e-9)).sum())
)
user_features = user_features.fillna(0)

try:
    from sklearn.ensemble import IsolationForest
    iso = IsolationForest(random_state=RANDOM_SEED, contamination=0.03)
    feature_cols = ['tweet_volume', 'token_count', 'hashtag_count', 'mention_count', 'url_count', 'unique_topic_count', 'stance_entropy']
    user_features['anomaly_flag'] = iso.fit_predict(user_features[feature_cols])
    user_features['anomaly_raw'] = iso.decision_function(user_features[feature_cols])
except Exception as e:
    print('IsolationForest unavailable:', e)
    user_features['anomaly_flag'] = 1
    user_features['anomaly_raw'] = 0.0

user_features['amplification_proxy'] = (
    (user_features['tweet_volume'].rank(pct=True) > 0.9).astype(int) +
    (user_features['hashtag_count'].rank(pct=True) > 0.9).astype(int) +
    (user_features['unique_topic_count'].rank(pct=True) < 0.3).astype(int) +
    (user_features['stance_entropy'].rank(pct=True) < 0.3).astype(int)
)

user_features = user_features.reset_index().rename(columns={'user_id': 'user'})
user_features['user'] = user_features['user'].astype(str)
user_features.sort_values(['amplification_proxy', 'anomaly_raw'], ascending=[False, True]).head(25).to_csv(
    OUTPUT_DIR / 'possible_amplification_accounts.csv', index=False
)

if user_graph.number_of_nodes() > 2:
    degree_centrality = nx.degree_centrality(user_graph)
    betweenness = nx.betweenness_centrality(user_graph, k=min(50, user_graph.number_of_nodes() - 1), seed=RANDOM_SEED)
    try:
        eigenvector = nx.eigenvector_centrality_numpy(user_graph)
    except Exception:
        eigenvector = {n: 0 for n in user_graph.nodes()}
    centrality_df = pd.DataFrame({
        'user': [str(n) for n in user_graph.nodes()],
        'degree_centrality': [degree_centrality.get(n, 0) for n in user_graph.nodes()],
        'betweenness': [betweenness.get(n, 0) for n in user_graph.nodes()],
        'eigenvector': [eigenvector.get(n, 0) for n in user_graph.nodes()],
    })
    centrality_df = centrality_df.merge(user_features, on='user', how='left')
    centrality_df.sort_values(['degree_centrality', 'betweenness'], ascending=False).to_csv(OUTPUT_DIR / 'centrality_ranking.csv', index=False)

polarization_report

{'community_assortativity': None,
 'community_count': 1544,
 'support_vs_oppose_sentiment_gap': 0.1188993414887267}

In [13]:
# %%
# Section 18b + 20b: Temporal anomalies and advanced slide content mapping
topic_col = 'semantic_topic' if 'semantic_topic' in df.columns else 'lda_topic'

if 'created_at' in df.columns and pd.api.types.is_datetime64_any_dtype(df['created_at']):
    daily = df.set_index('created_at').resample('D').agg({
        'tweet_id': 'count',
        'sentiment_score': 'mean'
    }).rename(columns={'tweet_id': 'volume'})
    daily['volume_7d_ma'] = daily['volume'].rolling(7, min_periods=1).mean()
    daily['volume_z'] = (daily['volume'] - daily['volume'].rolling(14, min_periods=7).mean()) / (daily['volume'].rolling(14, min_periods=7).std() + 1e-9)
    daily['volume_anomaly'] = daily['volume_z'].abs() > 2.5
    daily.to_csv(OUTPUT_DIR / 'daily_trends.csv')

    plt.figure(figsize=(12, 4))
    plt.plot(daily.index, daily['volume'], label='Daily volume', linewidth=1)
    plt.plot(daily.index, daily['volume_7d_ma'], label='7D moving average', linewidth=2)
    anomaly_idx = daily.index[daily['volume_anomaly'].fillna(False)]
    plt.scatter(anomaly_idx, daily.loc[anomaly_idx, 'volume'], color='red', s=15, label='Anomaly')
    plt.legend()
    plt.title('MBG discussion volume and anomalies')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'daily_volume_anomaly.png', dpi=220)
    plt.close()

    topic_week = df.assign(week=df['created_at'].dt.to_period('W').astype(str)).pivot_table(index='week', columns=topic_col, values='tweet_id', aggfunc='count', fill_value=0)
    topic_week.to_csv(OUTPUT_DIR / 'topic_weekly_mix.csv')

advanced_slides = {
    'background': {
        'title': 'Background',
        'bullets': [
            'Percakapan MBG di X menunjukkan dua arus besar: dukungan program dan kritik implementasi.',
            'Dataset berasal dari interaksi X yang disediakan panitia, tanpa data eksternal.',
            'Analisis diarahkan ke tema, aktor kunci, komunitas, sentimen, dan indikator koordinasi akun.'
        ],
        'figures': [str(OUTPUT_DIR / 'daily_volume_anomaly.png')]
    },
    'research_question': {
        'title': 'Research Questions',
        'bullets': [
            'Apa tema dominan dalam diskusi MBG dan bagaimana pergeserannya dari waktu ke waktu?',
            'Siapa aktor kunci dan komunitas apa yang menggerakkan percakapan?',
            'Seberapa polar wacana MBG dan apakah ada pola anomali yang menyerupai amplifikasi terkoordinasi?'
        ],
        'figures': []
    },
    'methodology': {
        'title': 'Methodology (1/2)',
        'bullets': [
            'Preprocessing Bahasa Indonesia, domain stopword, tokenisasi, dan fitur akun.',
            'Topic modelling semantik dengan BERTopic / fallback clustering berbasis embedding atau TF-IDF.'
        ],
        'figures': []
    },
    'methodology_2': {
        'title': 'Methodology (2/2)',
        'bullets': [
            'SNA pada graph mention/retweet dengan centrality, komunitas Louvain, dan assortativity.',
            'Anomaly detection pada level akun untuk mengidentifikasi pola amplifikasi yang tidak biasa.'
        ],
        'figures': []
    },
    'results': {
        'title': 'Results & Discussion',
        'bullets': [
            'Topik dukungan cenderung berfokus pada manfaat, gizi anak, dan pemerataan.',
            'Topik kritik menonjol pada keracunan, anggaran, kualitas menu, dan implementasi sekolah.',
            'Akun media/resmi dan influencer politik mendominasi jaringan, sementara sebagian akun menunjukkan pola amplifikasi tinggi.'
        ],
        'figures': [str(OUTPUT_DIR / 'top_hashtags.csv'), str(OUTPUT_DIR / 'centrality_ranking.csv'), str(OUTPUT_DIR / 'possible_amplification_accounts.csv')]
    },
    'important_findings': {
        'title': 'Important Findings',
        'bullets': [
            'Diskusi MBG terbelah antara narasi manfaat dan kritik risiko/anggaran.',
            'Komunitas jaringan membantu memetakan segmen pendukung, pengkritik, dan amplifikasi.',
            'Anomali volume dan pola akun memberi sinyal early warning untuk isu yang perlu respons cepat.'
        ],
        'figures': []
    },
    'suggestions': {
        'title': 'Suggestions',
        'bullets': [
            'Fokus komunikasi publik pada kualitas implementasi, transparansi anggaran, dan keamanan pangan.',
            'Gunakan akun/komunitas yang kredibel untuk meredam isu negatif dengan respons berbasis data.',
            'Bangun monitoring berkala atas lonjakan volume, topic shift, dan akun berindikasi amplifikasi terkoordinasi.'
        ],
        'figures': []
    },
    'ai_declaration': {
        'title': 'AI Usage Declaration',
        'bullets': [
            'AI dipakai untuk membantu penulisan kode, debugging, dan penyusunan kerangka analisis.',
            'Seluruh interpretasi substantif, validasi temuan, dan penyusunan akhir tetap harus diverifikasi oleh tim.'
        ],
        'figures': []
    }
}
with open(OUTPUT_DIR / 'slide_contents_advanced.json', 'w', encoding='utf-8') as f:
    json.dump(advanced_slides, f, ensure_ascii=False, indent=2)

print('Advanced slide mapping saved.')

Advanced slide mapping saved.


In [14]:
# %%
# Section 22: Dynamic narrative generator for slide-ready insights
report = {}

# Load generated artifacts if available
artifact_loaders = {
    'top_hashtags': OUTPUT_DIR / 'top_hashtags.csv',
    'top_users': OUTPUT_DIR / 'top_users.csv',
    'top_ngrams': OUTPUT_DIR / 'top_ngrams.csv',
    'lda_topics': OUTPUT_DIR / 'lda_topics.json',
    'tweet_topics': OUTPUT_DIR / 'tweet_topics.csv',
    'centrality_ranking': OUTPUT_DIR / 'centrality_ranking.csv',
    'possible_amplification_accounts': OUTPUT_DIR / 'possible_amplification_accounts.csv',
    'polarization_report': OUTPUT_DIR / 'polarization_report.json',
    'community_sentiment': OUTPUT_DIR / 'community_sentiment.csv',
}

if artifact_loaders['top_hashtags'].exists():
    top_hashtags_df = pd.read_csv(artifact_loaders['top_hashtags'])
    report['top_hashtag_1'] = top_hashtags_df.iloc[0]['hashtag'] if len(top_hashtags_df) else None
    report['top_hashtag_2'] = top_hashtags_df.iloc[1]['hashtag'] if len(top_hashtags_df) > 1 else None

if artifact_loaders['top_users'].exists():
    top_users_df = pd.read_csv(artifact_loaders['top_users'])
    report['top_user_1'] = str(top_users_df.iloc[0]['user_id']) if len(top_users_df) else None
    report['top_user_1_count'] = int(top_users_df.iloc[0]['count']) if len(top_users_df) else 0

if artifact_loaders['top_ngrams'].exists():
    top_ngrams_df = pd.read_csv(artifact_loaders['top_ngrams'])
    report['top_ngram_1'] = top_ngrams_df.iloc[0]['ngram'] if len(top_ngrams_df) else None
    report['top_ngram_2'] = top_ngrams_df.iloc[1]['ngram'] if len(top_ngrams_df) > 1 else None

if artifact_loaders['centrality_ranking'].exists():
    centrality_df = pd.read_csv(artifact_loaders['centrality_ranking'])
    if len(centrality_df):
        report['top_central_user'] = str(centrality_df.iloc[0]['user'])
        report['top_central_degree'] = float(centrality_df.iloc[0].get('degree_centrality', 0))

if artifact_loaders['possible_amplification_accounts'].exists():
    amp_df = pd.read_csv(artifact_loaders['possible_amplification_accounts'])
    report['possible_amplification_count'] = int(len(amp_df))
    report['possible_amplification_user_1'] = str(amp_df.iloc[0]['user']) if len(amp_df) else None

if artifact_loaders['polarization_report'].exists():
    with open(artifact_loaders['polarization_report'], 'r', encoding='utf-8') as f:
        polar = json.load(f)
    report.update(polar)

# Build dynamic prose for slide notes
insights = []
if report.get('top_hashtag_1'):
    insights.append(f"Hashtag paling dominan adalah {report['top_hashtag_1']}, dengan {report.get('top_hashtag_2', 'hashtag lain')} sebagai penguat narasi publik.")
if report.get('top_user_1'):
    insights.append(f"Akun paling aktif adalah {report['top_user_1']} dengan {report['top_user_1_count']} tweet, menunjukkan konsentrasi aktivitas pada segelintir akun.")
if report.get('top_central_user'):
    insights.append(f"Aktor jaringan paling sentral adalah {report['top_central_user']}, sehingga layak dijadikan fokus pembacaan aliran informasi.")
if report.get('community_assortativity') is not None:
    insights.append(f"Indikator assortativity komunitas sebesar {report['community_assortativity']:.3f} mengindikasikan tingkat pengelompokan jaringan yang {('cukup tinggi' if abs(report['community_assortativity']) > 0.1 else 'rendah')}.")
if report.get('possible_amplification_count'):
    insights.append(f"Terdeteksi {report['possible_amplification_count']} akun dengan pola amplifikasi yang patut diawasi sebagai proxy buzzer/coordinated behavior.")

summary_md = [
    '# Dynamic Findings Summary',
    '',
    '## Auto-generated narrative',
]
summary_md.extend([f'- {x}' for x in insights])
summary_md.extend([
    '',
    '## Use in slides',
    '- Background: gunakan 1 kalimat dari ringkasan ini + grafik tren volume.',
    '- Results: gunakan top hashtags, top actors, dan komunitas jaringan.',
    '- Findings: gunakan indikasi polaritas dan akun amplifikasi sebagai early warning.',
])

with open(OUTPUT_DIR / 'dynamic_findings_summary.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_md))

with open(OUTPUT_DIR / 'dynamic_findings_summary.json', 'w', encoding='utf-8') as f:
    json.dump({'report': report, 'insights': insights}, f, ensure_ascii=False, indent=2)

print('\n'.join(summary_md[:12]))

# Dynamic Findings Summary

## Auto-generated narrative
- Hashtag paling dominan adalah #makanbergizigratis, dengan #mbg sebagai penguat narasi publik.
- Akun paling aktif adalah 1720665183188922368 dengan 167 tweet, menunjukkan konsentrasi aktivitas pada segelintir akun.
- Aktor jaringan paling sentral adalah prabowo, sehingga layak dijadikan fokus pembacaan aliran informasi.
- Terdeteksi 25 akun dengan pola amplifikasi yang patut diawasi sebagai proxy buzzer/coordinated behavior.

## Use in slides
- Background: gunakan 1 kalimat dari ringkasan ini + grafik tren volume.
- Results: gunakan top hashtags, top actors, dan komunitas jaringan.
- Findings: gunakan indikasi polaritas dan akun amplifikasi sebagai early warning.
